# 01 — Limpeza de dados

Objetivo: transformar a base bruta (`data/raw/marketing_campaign.csv`) numa base confiável para análise, tratando os problemas mapeados no Dia 1 e registrando cada decisão.


In [1]:
import pandas as pd
import numpy as np

# sep="\t" porque o arquivo é separado por tabulação
df = pd.read_csv("../data/raw/marketing_campaign.csv", sep="\t")

print(df.shape)   # (linhas, colunas)
df.head()

(2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [2]:
df.info()           # tipo de cada coluna e quantos valores não nulos
df.describe().T     # estatísticas: média, mínimo, máximo, quartis

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

,count,mean,std,min,25%,50%,75%,max
ID,2240.0,5592.159821,3246.662198,0.0,2828.25,5458.5,8427.75,11191.0
Year_Birth,2240.0,1968.805804,11.984069,1893.0,1959.00,1970.0,1977.00,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,35303.00,51381.5,68522.00,666666.0
Kidhome,2240.0,0.444196,0.538398,0.0,0.00,0.0,1.00,2.0
Teenhome,2240.0,0.506250,0.544538,0.0,0.00,0.0,1.00,2.0
Recency,2240.0,49.109375,28.962453,0.0,24.00,49.0,74.00,99.0
MntWines,2240.0,303.935714,336.597393,0.0,23.75,173.5,504.25,1493.0
MntFruits,2240.0,26.302232,39.773434,0.0,1.00,8.0,33.00,199.0
MntMeatProducts,2240.0,166.950000,225.715373,0.0,16.00,67.0,232.00,1725.0
MntFishProducts,2240.0,37.525446,54.628979,0.0,3.00,12.0,50.00,259.0


**Inspeção inicial:** a base tem 2.240 linhas e 29 colunas. Os problemas mapeados no Dia 1 se confirmam: 24 valores nulos em `Income`, `Year_Birth` mínimo de 1893, `Income` máximo de 666.666, `Dt_Customer` lido como texto e `Z_CostContact`/`Z_Revenue` constantes.

## Passo 2 — Diagnóstico de qualidade

In [3]:
# Valores nulos por coluna
print("=== Nulos ===")
print(df.isna().sum()[df.isna().sum() > 0])

# Linhas duplicadas, ignorando o ID (mesmo cliente cadastrado duas vezes?)
cols_sem_id = [c for c in df.columns if c != "ID"]
print("\n=== Duplicados (sem considerar ID) ===")
print(df.duplicated(subset=cols_sem_id).sum())

# Categorias de texto
print("\n=== Marital_Status ===")
print(df["Marital_Status"].value_counts())
print("\n=== Education ===")
print(df["Education"].value_counts())

# Colunas constantes (só um valor)
print("\n=== Colunas constantes ===")
print(df.nunique()[df.nunique() == 1])

=== Nulos ===
Income    24
dtype: int64

=== Duplicados (sem considerar ID) ===
182

=== Marital_Status ===
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

=== Education ===
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

=== Colunas constantes ===
Z_CostContact    1
Z_Revenue        1
dtype: int64


In [4]:
# Conferência dos problemas mapeados no Dia 1
print("=== Problemas mapeados no Dia 1 ===")
print("Year_Birth antes de 1901:", (df["Year_Birth"] < 1901).sum())
print("Marital_Status inválidos (Absurd, YOLO):", df["Marital_Status"].isin(["Absurd", "YOLO"]).sum())
print("Marital_Status = Alone:", (df["Marital_Status"] == "Alone").sum())
print("Income vazio:", df["Income"].isna().sum())
print("Income = 666666:", (df["Income"] == 666666).sum())
print("Income = 7500:", (df["Income"] == 7500).sum())
print("Income abaixo de 5.000:", (df["Income"] < 5000).sum())
print("Income entre 150 mil e 162 mil:", df["Income"].between(150000, 162397).sum())

# Clientes sem nenhuma compra (quebram ticket médio e mix de canal)
total_compras = df["NumWebPurchases"] + df["NumCatalogPurchases"] + df["NumStorePurchases"]
print("\n=== Checagem para os KPIs ===")
print("Clientes com zero compras:", (total_compras == 0).sum())

=== Problemas mapeados no Dia 1 ===
Year_Birth antes de 1901: 3
Marital_Status inválidos (Absurd, YOLO): 4
Marital_Status = Alone: 3
Income vazio: 24
Income = 666666: 1
Income = 7500: 12
Income abaixo de 5.000: 6
Income entre 150 mil e 162 mil: 7

=== Checagem para os KPIs ===
Clientes com zero compras: 6


In [5]:
# Quem são os clientes sem nenhuma compra?
mnt_cols = [c for c in df.columns if c.startswith("Mnt")]
sem_compras = df[total_compras == 0]
sem_compras[["ID", "Income", "Year_Birth", "Recency"] + mnt_cols + ["NumDealsPurchases", "Response"]]

,ID,Income,Year_Birth,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,Response
655,5555,153924.0,1975,81,1,1,1,1,1,1,0,0
981,3955,4861.0,1965,20,2,1,1,1,0,1,0,0
1245,6862,1730.0,1971,65,1,1,3,1,1,1,15,0
1524,11110,3502.0,1973,56,2,1,1,0,0,1,0,0
1846,9931,4023.0,1963,29,5,0,1,1,1,1,15,0
2132,11181,156924.0,1949,85,2,1,2,1,1,1,0,0


## Passo 3 — Aplicar a limpeza

Cada regra está numa célula e registra no `log` quantas linhas saíram. As decisões seguem a tabela "Problemas de qualidade e decisões de limpeza" do `docs/plan.md`.

In [6]:
# Lista que guarda cada decisão tomada, e o tamanho inicial da base
log = []
linhas_inicio = len(df)
print("Linhas no início:", linhas_inicio)

Linhas no início: 2240


### 3.1 Remover colunas constantes
`Z_CostContact` e `Z_Revenue` têm o mesmo valor em todas as linhas e não carregam informação.

In [7]:
df = df.drop(columns=["Z_CostContact", "Z_Revenue"])
log.append("Removidas Z_CostContact e Z_Revenue: valor constante.")
print("Colunas agora:", df.shape[1])

Colunas agora: 27


### 3.2 Remover duplicados
Linhas idênticas em todas as colunas, exceto `ID`, são o mesmo cliente cadastrado duas vezes. Mantemos a primeira ocorrência.

In [8]:
antes = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "ID"])
log.append(f"Removidos {antes - len(df)} registros duplicados (mesmos dados, IDs diferentes).")
print(log[-1])

Removidos 182 registros duplicados (mesmos dados, IDs diferentes).


In [9]:
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
print(df["Dt_Customer"].dtype)
print("Período:", df["Dt_Customer"].min().date(), "a", df["Dt_Customer"].max().date())

datetime64[us]
Período: 2012-07-30 a 2014-06-29


### 3.4 Criar idade e remover erros de cadastro
A idade é calculada com o ano do último cadastro da base como referência, e não com o ano atual: os dados são de 2012 a 2014, e calcular com 2026 inflaria todas as idades. Clientes com mais de 100 anos são erro de cadastro.

In [10]:
ano_ref = df["Dt_Customer"].dt.year.max()
df["Idade"] = ano_ref - df["Year_Birth"]
print("Ano de referência:", ano_ref)

antes = len(df)
df = df[df["Idade"] <= 100].copy()
log.append(f"Removidos {antes - len(df)} clientes com idade acima de 100 anos (erro de cadastro).")
print(log[-1])
print("Idade mínima e máxima:", df["Idade"].min(), "e", df["Idade"].max())

Ano de referência: 2014
Removidos 3 clientes com idade acima de 100 anos (erro de cadastro).
Idade mínima e máxima: 18 e 74


### 3.5 Renda: remover nulos e outlier extremo
Decisão: remover as rendas vazias em vez de preencher com a mediana. As hipóteses H1 e H6 dependem diretamente da renda, e preencher colocaria esses clientes artificialmente no meio da distribuição. A perda é de cerca de 1% da base. O valor 666.666 é um outlier impossível e também sai.

In [11]:
antes = len(df)
df = df[df["Income"].notna()].copy()
log.append(f"Removidos {antes - len(df)} clientes com renda vazia.")
print(log[-1])

antes = len(df)
df = df[df["Income"] < 200_000].copy()
log.append(f"Removidos {antes - len(df)} clientes com renda acima de 200 mil (outlier).")
print(log[-1])

Removidos 24 clientes com renda vazia.
Removidos 1 clientes com renda acima de 200 mil (outlier).


### 3.6 Estado civil: agrupar em duas categorias de negócio
"Mora junto" (Married, Together) e "Mora sozinho" (Single, Divorced, Widow, Alone). Absurd e YOLO são inválidos e saem. Esse agrupamento atende à hipótese H8.

In [12]:
mapa_civil = {
    "Married": "Mora junto", "Together": "Mora junto",
    "Single": "Mora sozinho", "Divorced": "Mora sozinho",
    "Widow": "Mora sozinho", "Alone": "Mora sozinho",
}
antes = len(df)
df = df[df["Marital_Status"].isin(mapa_civil.keys())].copy()  # remove "Absurd" e "YOLO"
df["Estado_Civil"] = df["Marital_Status"].map(mapa_civil)
log.append(f"Removidos {antes - len(df)} registros com estado civil inválido (Absurd, YOLO).")
print(log[-1])
print(df["Estado_Civil"].value_counts())

Removidos 4 registros com estado civil inválido (Absurd, YOLO).
Estado_Civil
Mora junto      1300
Mora sozinho     726
Name: count, dtype: int64


### 3.7 Escolaridade em português
2n Cycle equivale a mestrado no sistema europeu e é agrupado com Master.

In [13]:
mapa_edu = {"Basic": "Básico", "Graduation": "Graduação",
            "2n Cycle": "Mestrado", "Master": "Mestrado", "PhD": "Doutorado"}
df["Escolaridade"] = df["Education"].map(mapa_edu)
print(df["Escolaridade"].value_counts())
print("Sem tradução:", df["Escolaridade"].isna().sum())

Escolaridade
Graduação    1017
Mestrado      518
Doutorado     442
Básico         49
Name: count, dtype: int64
Sem tradução: 0


### 3.8 Remover clientes sem nenhuma compra
Identificado no diagnóstico: clientes com zero compras nos canais (web, catálogo, loja), mas com gasto nas colunas `Mnt`, o que é contraditório. Eles também concentram rendas extremas. Além de inconsistentes, quebrariam o ticket médio e o mix de canal (divisão por zero).

In [14]:
total_compras = df["NumWebPurchases"] + df["NumCatalogPurchases"] + df["NumStorePurchases"]

antes = len(df)
df = df[total_compras > 0].copy()
log.append(f"Removidos {antes - len(df)} clientes sem nenhuma compra nos canais (registros inconsistentes).")
print(log[-1])

Removidos 6 clientes sem nenhuma compra nos canais (registros inconsistentes).


## Validação da limpeza
Checagens automáticas: se todas passarem, a base está pronta para a análise.

In [15]:
print("=== Validação ===")
print("Nulos restantes:", df.isna().sum().sum())
print("Duplicados restantes:", df.duplicated(subset=[c for c in df.columns if c != "ID"]).sum())
print("Idade máxima:", df["Idade"].max())
print("Renda máxima:", df["Income"].max())
print("Estado civil:", df["Estado_Civil"].unique())
print("Clientes sem compras:", ((df["NumWebPurchases"] + df["NumCatalogPurchases"] + df["NumStorePurchases"]) == 0).sum())

=== Validação ===
Nulos restantes: 0
Duplicados restantes: 0
Idade máxima: 74
Renda máxima: 162397.0
Estado civil: <StringArray>
['Mora sozinho', 'Mora junto']
Length: 2, dtype: str
Clientes sem compras: 0


## Log de decisões

In [16]:
print(f"Linhas no início: {linhas_inicio}")
for item in log:
    print("-", item)
print(f"Linhas no final: {len(df)}")
print(f"Removidas no total: {linhas_inicio - len(df)} ({(linhas_inicio - len(df)) / linhas_inicio:.1%} da base)")

Linhas no início: 2240
- Removidas Z_CostContact e Z_Revenue: valor constante.
- Removidos 182 registros duplicados (mesmos dados, IDs diferentes).
- Removidos 3 clientes com idade acima de 100 anos (erro de cadastro).
- Removidos 24 clientes com renda vazia.
- Removidos 1 clientes com renda acima de 200 mil (outlier).
- Removidos 4 registros com estado civil inválido (Absurd, YOLO).
- Removidos 6 clientes sem nenhuma compra nos canais (registros inconsistentes).
Linhas no final: 2020
Removidas no total: 220 (9.8% da base)
